In [6]:
%%capture
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:

# 1. Read the dataset
file_path = '/kaggle/input/datasets/bethelhemalemayehu/welfake/WELFake_Dataset.csv'
df = pd.read_csv(file_path)

# 2. Display the first few rows
print(df.head())

# 3. Check the number of fake vs real news samples
# 0 = Real, 1 = Fake
print(df['label'].value_counts())

# IMPORTANT: Remove rows with missing titles
df = df.dropna(subset=['title'])

   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  
label
1    37106
0    35028
Name: count, dtype: int64


In [8]:
# Text Preprocessing

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf.fit_transform(df['title']).toarray()
y = df['label'].values

In [9]:
# Split the Dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Build the Neural Network

model = Sequential([
    Dense(128, activation='relu', input_shape=(5000,)), # Input -> 128
    Dense(64, activation='relu'),                       # Hidden -> 64
    Dense(1, activation='sigmoid')                      # Output (Fake/Real)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1773754818.435547      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773754818.442049      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [11]:
# Train the model

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train for 10-20 epochs as per instructions
history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/15


I0000 00:00:1773754824.604870     126 service.cc:152] XLA service 0x7b0334004570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773754824.604909     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773754824.604913     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773754825.039182     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


  60/1790 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.5912 - loss: 0.6772

I0000 00:00:1773754826.194165     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1790/1790 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.8376 - loss: 0.3620 - val_accuracy: 0.8908 - val_loss: 0.2624
Epoch 2/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9155 - loss: 0.2073 - val_accuracy: 0.9003 - val_loss: 0.2488
Epoch 3/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9459 - loss: 0.1393 - val_accuracy: 0.9030 - val_loss: 0.2676
Epoch 4/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9753 - loss: 0.0707 - val_accuracy: 0.9083 - val_loss: 0.3409
Epoch 5/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9926 - loss: 0.0247 - val_accuracy: 0.9051 - val_loss: 0.4765
Epoch 6/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9980 - loss: 0.0079 - val_accuracy: 0.9064 - val_loss: 0.5818
Epoch 7/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9991 - loss: 0.0037 - val_accuracy: 0.9060 - val_loss: 0.6603
Epoch 8/15
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9992 - loss: 0.0036 - val_accura

In [12]:
# Evaluate and Test

model.evaluate(X_test, y_test)

# Manual Test
def predict_news(headline):
    vec = tfidf.transform([headline]).toarray()
    pred = model.predict(vec)[0][0]
    return "Fake News" if pred >= 0.5 else "Real News"

print(predict_news("Scientists confirm water on Mars."))

448/448 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9060 - loss: 0.9122
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step
Fake News


In [13]:
model.save('fake_news_model.h5')